# Extract features (single subject)

Feature extraction (for later training and/or classifying).

In [1]:
%load_ext autoreload
%autoreload 2

from pyfix import extract
from pyfix.util import SegType
import os.path as op

datadir = '../../tests/data'

In [2]:
dhcpdir = f'{datadir}/sub-CC00108XX09/ses-36800'

features = extract.extract_features(
    fixdir=f'{datadir}/testing-pyfix/pyfix',
    func=op.join(dhcpdir, 'ica/func_filtered.nii.gz'),
    func_brainmask=op.join(dhcpdir, 'fix/mask.nii.gz'),
    struct=op.join(dhcpdir, 'fix/reg/highres.nii.gz'),
    struct2func_xfm=op.join(dhcpdir, 'fix/reg/highres2example_func.mat'),
    struct_dseg=op.join(dhcpdir, 'fix/reg/highres_pveseg.nii.gz'),
    ic_timeseries=op.join(dhcpdir, 'ica/melodic_mix'),
    ic_map=op.join(dhcpdir, 'ica/melodic_IC.nii.gz'),
    mot_param=op.join(dhcpdir, 'mcdc/func_mcdc.eddy_parameters'),
    struct_dseg_type=SegType.fsl_fast,
    hp_fwhm=150,
)

/Users/seanf/dev/pyfix/pyfix/feature.py:433: RuntimeWarning: invalid value encountered in log
  lmbda = -(1 / deltat) * np.log((Sxy - mu * Sx - mu * Sy + n * mu ** 2) / (Sxx - 2 * mu * Sx + n * mu ** 2))


## Extract features from FEAT dir

Extract features from files in a FEAT directory where ICA has been run.

In [ ]:
featdir = '/Users/seanf/data/fsldata/Rest_MB6.feat'

features = extract.from_feat_dir(featdir)

## Custom feature extractor/s

Add custom feature extractors.  A custom feature extractor is a method that is decorated with the `@feature_extractor` decorator, accepts the the pyfix data dictionary as input (to be replaced with a dataclass), and returns a feature.  Allowed return types are list, scalar, dictionary, and pandas `Series`.

In [ ]:
from pyfix.feature import feature_extractor
import pandas as pd

@feature_extractor
def grot(data):
    return [1, 2, 3]

@feature_extractor
def grot2(data):
    return 4

@feature_extractor
def grot3(data):
    return {'poo': 5}

@feature_extractor
def grot4(data):
    return pd.Series({'pop': 5})

dhcpdir = '/Users/seanf/data/neofmri_2nd_release_rerun2/sub-CC00108XX09/ses-36800'

features = extract.extract_features(
    fixdir='/Users/seanf/scratch/testing-pyfix/pyfix',
    func=op.join(dhcpdir, 'ica/func_filtered.nii.gz'),
    func_brainmask=op.join(dhcpdir, 'fix/mask.nii.gz'),
    struct=op.join(dhcpdir, 'fix/reg/highres.nii.gz'),
    struct2func_xfm=op.join(dhcpdir, 'fix/reg/highres2example_func.mat'),
    struct_dseg=op.join(dhcpdir, 'fix/reg/highres_pveseg.nii.gz'),
    ic_timeseries=op.join(dhcpdir, 'ica/melodic_mix'),
    ic_map=op.join(dhcpdir, 'ica/melodic_IC.nii.gz'),
    mot_param=op.join(dhcpdir, 'mcdc/func_mcdc.eddy_parameters'),
    struct_dseg_type=SegType.fsl_fast,
    hp_fwhm=150,
)

In [ ]:
features.head()

# Train (group)
Train FIX on a group of labelled subjects.

Generate a table of files from legacy FIX directories.

In [1]:
from pyfix import legacy, io, classify
import glob

file_table = legacy.generate_file_table(*glob.glob('/Users/seanf/data/HCP/fix_labels/*.ica'))
file_table.head()

,features,labels,subid
0,/Users/seanf/data/HCP/fix_labels/36.ica/fix/fe...,/Users/seanf/data/HCP/fix_labels/36.ica/hand_l...,0
1,/Users/seanf/data/HCP/fix_labels/22.ica/fix/fe...,/Users/seanf/data/HCP/fix_labels/22.ica/hand_l...,1
2,/Users/seanf/data/HCP/fix_labels/23.ica/fix/fe...,/Users/seanf/data/HCP/fix_labels/23.ica/hand_l...,2
3,/Users/seanf/data/HCP/fix_labels/37.ica/fix/fe...,/Users/seanf/data/HCP/fix_labels/37.ica/hand_l...,3
4,/Users/seanf/data/HCP/fix_labels/21.ica/fix/fe...,/Users/seanf/data/HCP/fix_labels/21.ica/hand_l...,4


Generate a `feature_table` that collates all the features from the files specified in the `file_table`.

In [2]:
training_data = classify.collate_features(file_table, legacy_features=True)
training_data.head()

,nic,arvswgn:0,arvswgn:1,arfull:0,arfull:1,arfull:2,arfull:3,arfull:4,skewness,kurtosis,...,dim:0,dim:1,dim:2,dim:3,pixdim:0,pixdim:1,pixdim:2,pixdim:3,group,label
0,247,-0.004475,0.042334,0.038705,-0.98044,0.038703,-0.98781,0.007513,0.328840,2.6123,...,91,109,91,1200,2,2,2,0.72,0,True
1,247,-0.004769,0.036614,0.032718,-0.98349,0.032699,-1.00760,0.024518,-0.326090,2.0584,...,91,109,91,1200,2,2,2,0.72,0,True
2,247,-0.016596,0.132840,0.119340,-0.93838,0.118980,-0.98998,0.054986,-0.128070,2.1478,...,91,109,91,1200,2,2,2,0.72,0,True
3,247,-0.012649,0.101670,0.092065,-0.95282,0.091779,-1.00590,0.055716,0.295100,2.2495,...,91,109,91,1200,2,2,2,0.72,0,True
4,247,-0.009742,0.091605,0.081953,-0.95811,0.081886,-0.93078,-0.028525,0.024646,1.9019,...,91,109,91,1200,2,2,2,0.72,0,True


Train the classifier with LOO cross-validation for assessing generalisation error.  Any classifier that conforms to `scikit-learn` API can be used.

In [5]:
from sklearn import svm, ensemble, neighbors, tree, neural_network as nn

# clf = classify.FixLegacyClassifier()
clf = svm.SVC(probability=False, class_weight="balanced", gamma='scale', kernel='linear')
# clf = svm.SVC(probability=False, class_weight="balanced", gamma='scale', kernel='rbf')
# clf = ensemble.RandomForestClassifier(n_estimators=50)
# clf = nn.MLPClassifier(solver='lbfgs', early_stopping=True)
# clf = ensemble.GradientBoostingClassifier()

clf = classify.train(training_data, loo=True)


	TPR	TNR	(3*TPR+TNR)/4
Mean	99.51	99.20	99.43
Median	100.00	99.50	99.80
Pooled	99.48	99.22	99.42

Pooled confusion matrix:
[[ 2314    12]
 [  161 20390]]


# Classify (single subject)

Apply a trained model to classify the independent components from a single subject.

In [ ]:
legacy.read_labels(file_table.loc[0, 'labels'])

In [ ]:
from pyfix import classify
    
features0 = file_table.loc[0, 'features']
    
Y = classify.classify(features0, clf)
Y = np.where(Y==1)[0]+1

print(Y)

# Apply (single subject)

Apply cleanup by regressing ICs labelled/classified as noise by FIX from the data.

In [ ]:
with open('/Users/seanf/data/neofmri_2nd_release_rerun2/sub-CC00108XX09/ses-36800/fix/fix4melview_train35_thr10.txt', 'r') as f:
    ddremove = f.readlines()[-1]

ddremove = np.fromstring(ddremove.strip('[]\n'), dtype=int, sep=',')

print(ddremove)

In [ ]:
from pyfix import apply

apply.apply(
    funcname='/Users/seanf/data/neofmri_2nd_release_rerun2/sub-CC00108XX09/ses-36800/ica/func_filtered.nii.gz', 
    icamix='/Users/seanf/data/neofmri_2nd_release_rerun2/sub-CC00108XX09/ses-36800/ica/melodic_mix', 
    ddremove=np.array(ddremove), 
    temporal_fwhm=150, 
    motparams=None,
    filt_func=False, 
    filt_motparams=True, 
    func_tr=0.392, 
    outname='/Users/seanf/scratch/testing-pyfix/pyfix/func_clean.nii.gz',
)

In [ ]:
! tree /Users/seanf/scratch/testing-pyfix/pyfix/